# Proyecto de Calidad de Datos
## 1. CARGA DE DATOS

In [1]:
import pandas as pd

df = pd.read_csv("transactions_raw.csv")
df = df.copy()

# Exploración inicial de datos

In [2]:
df.head()
df.info()
df.isna().sum() #Cantidad de nulos por columna
df.duplicated().sum() # para ver cuantos duplicados hay
df[df.duplicated(keep=False)] # Cual es el duplicado

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    21 non-null     int64 
 1   user_id           20 non-null     object
 2   transaction_date  20 non-null     object
 3   amount            21 non-null     object
 4   currency          21 non-null     object
 5   status            21 non-null     object
 6   payment_method    20 non-null     object
 7   country           21 non-null     object
 8   created_at        20 non-null     object
 9   source_system     21 non-null     object
dtypes: int64(1), object(9)
memory usage: 1.8+ KB


,transaction_id,user_id,transaction_date,amount,currency,status,payment_method,country,created_at,source_system
3,1004,U004,2024-01-06,15,EUR,pending,transfer,ES,2024-01-06 09:30:00,partner_b
4,1004,U004,2024-01-06,15,EUR,pending,transfer,ES,2024-01-06 09:30:00,partner_b


# 2 — NORMALIZACIÓN

In [3]:
# Campo amount:
df["amount_numeric"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

# Campo transaction_date:
#transaction_date
df["transaction_date_parsed"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

# Campo created_at:
df["created_at_parsed"] = pd.to_datetime(
    df["created_at"],
    errors="coerce"
)

# Campo status:
df["status_normalized"] = df["status"].str.strip().str.lower()
df["status_normalized"].value_counts()

status_normalized
approved    16
pending      2
failed       2
reversed     1
Name: count, dtype: int64

# 3 — REGLAS PARA ESTABLECER TRANSACCIONES VÁLIDAS

In [4]:
# Regla 1 transaction_id:
invalid_tx_id = df["transaction_id"].isna()
invalid_tx_id.sum()

# Regla 2 user_id:
invalid_user = df["user_id"].isna()
invalid_user.sum()

# 3 transaction_date:
invalid_tx_date = df["transaction_date_parsed"].isna()
invalid_tx_date.sum()

# Regla 4 amount:
invalid_amount = (
    df["amount_numeric"].isna() |
    (df["amount_numeric"] == 0) |
    ((df["amount_numeric"] < 0) & (df["status_normalized"] == "approved"))
)

# Regla 5 created_at:
invalid_created = df["created_at_parsed"].isna()

# TRANSACCIONES VÁLIDAS

In [5]:
df["rule_user_valid"] = ~invalid_user
df["rule_amount_valid"] = ~invalid_amount
df["rule_tx_date_valid"] = ~invalid_tx_date
df["rule_created_valid"] = ~invalid_created
df["rule_not_duplicate"] = ~df.duplicated(keep=False)
df["rule_tx_id_valid"] = ~invalid_tx_id

# CREAMOS COLUMNA PARA SABER QUE TRANSACCIONES SON VÁLIDAS

In [6]:
df["is_valid"] = (
    df["rule_user_valid"] &
    df["rule_amount_valid"] &
    df["rule_tx_date_valid"] &
    df["rule_created_valid"] &
    df["rule_not_duplicate"] &
	df["rule_tx_id_valid"]
)

# TRANSACCIONES INVÁLIDAS CON SUS REGLAS

In [7]:
df.loc[df["is_valid"] == False, [
    "transaction_id",
    "rule_user_valid",
    "rule_amount_valid",
    "rule_tx_date_valid",
    "rule_created_valid",
    "rule_not_duplicate",
	"rule_tx_id_valid"
]]

,transaction_id,rule_user_valid,rule_amount_valid,rule_tx_date_valid,rule_created_valid,rule_not_duplicate,rule_tx_id_valid
1,1002,True,False,False,True,True,True
3,1004,True,True,True,True,False,True
4,1004,True,True,True,True,False,True
5,1005,False,True,True,True,True,True
6,1006,True,False,True,True,True,True
7,1007,True,True,False,True,True,True
8,1008,True,True,False,True,True,True
11,1011,True,False,True,True,True,True
12,1012,True,True,True,False,True,True
17,1017,True,True,False,True,True,True


# 4 — DATASET LIMPIO

In [8]:
valid_df = df[df["is_valid"] == True]
valid_df.shape
valid_df = valid_df.copy()

# 5 — ANOMALÍAS

In [18]:
# CONTROL DE RIESGO, con percentiles
p25 = valid_df["amount_numeric"].quantile(0.25)
p75 = valid_df["amount_numeric"].quantile(0.75)

# DEFINICION de umbral para alerta
amount_outlier = valid_df["amount_numeric"] > p75 * 3 #valores por encima de 3 veces el percentil 75

# flag de anomalias
df["flag_amount_outlier"] = False

df.loc[valid_df.index, "flag_amount_outlier"] = amount_outlier

## Cuantas alertas o anomalias hay?

In [10]:
valid_df["flag_amount_outlier"].value_counts()

flag_amount_outlier
False    9
True     1
Name: count, dtype: int64

## Cuales transacciones son atipicas?

In [11]:
valid_df.loc[
    valid_df["flag_amount_outlier"],
    ["transaction_id", "user_id", "amount_numeric", "status_normalized"]
]

,transaction_id,user_id,amount_numeric,status_normalized
15,1015,U015,99999.0,approved


# Analisis por usuario:

In [12]:
user_stats = (
    valid_df
    .groupby("user_id")["amount_numeric"]
    .agg(["count", "mean", "sum"])
    .sort_values("sum", ascending=False)
)

# 6 — MÉTRICAS

In [13]:
# Métricas globales:
total_rows = len(df)
valid_rows = df["is_valid"].sum()
invalid_rows = (~df["is_valid"]).sum()
pct_valid = df["is_valid"].mean()
pct_invalid = 1 - pct_valid

# Construcción del df para kPIs:

In [14]:
data_quality_global = pd.DataFrame({
    "metric_name": [
        "total_rows",
        "valid_rows",
        "invalid_rows",
        "pct_valid",
        "pct_invalid"
    ],
    "value": [
        total_rows,
        valid_rows,
        invalid_rows,
        pct_valid,
        pct_invalid
    ]
})

data_quality_global

,metric_name,value
0,total_rows,21.00000
1,valid_rows,10.00000
2,invalid_rows,11.00000
3,pct_valid,0.47619
4,pct_invalid,0.52381


## Calidad por regla:

In [15]:
user_id_invalid = (~df["rule_user_valid"]).sum()
transaction_date_invalid = (~df["rule_tx_date_valid"]).sum()
amount_invalid = (~df["rule_amount_valid"]).sum()
created_at_invalid = (~df["rule_created_valid"]).sum()
duplicate = (~df["rule_not_duplicate"]).sum()

# Construcción del df para kPIs de calidad de reglas:

In [16]:
rules_quality_global = pd.DataFrame({
    "metric_name": [
        "user_id_invalid",
        "transaction_date_invalid",
        "amount_invalid",
        "created_at_invalid",
        "duplicate"
    ],
    "value": [
        user_id_invalid,
        transaction_date_invalid,
        amount_invalid,
        created_at_invalid,
        duplicate
    ]
})

rules_quality_global

,metric_name,value
0,user_id_invalid,1
1,transaction_date_invalid,4
2,amount_invalid,3
3,created_at_invalid,2
4,duplicate,2


# 7 — DATASETS PARA REPORTING / BI

In [19]:
# Dataset para reporting (solo válidas y aprobadas)
reporting_df = (
    df[
        (df["is_valid"]) &
        (df["status_normalized"] == "approved")
    ]
    .copy()
)

# Dataset de anomalías
anomalies_df = df[
    (~df["is_valid"]) |
    (df["flag_amount_outlier"])
].copy()

# Dataset de auditoría
audit_df = df.copy()

## Exportación de resultados

In [20]:
reporting_df.to_csv("transactions_reporting.csv", index=False)
anomalies_df.to_csv("transactions_anomalies.csv", index=False)

# 8 — DATA QUALITY SCORE

In [21]:
dq_score = 1 - (invalid_rows / total_rows)
dq_score

np.float64(0.47619047619047616)

## Análisis dinámico de reglas

In [22]:
rule_cols = [col for col in df.columns if col.startswith("rule_")]

broken_counts = (~df[rule_cols]).sum()

rules_summary = broken_counts.reset_index()
rules_summary.columns = ["rule_name", "broken_count"]

rules_summary["pct_of_total"] = rules_summary["broken_count"] / total_rows
rules_summary["pct_of_invalid"] = rules_summary["broken_count"] / invalid_rows

rules_summary.sort_values(by="broken_count", ascending=False)

,rule_name,broken_count,pct_of_total,pct_of_invalid
2,rule_tx_date_valid,4,0.190476,0.363636
1,rule_amount_valid,3,0.142857,0.272727
4,rule_not_duplicate,2,0.095238,0.181818
3,rule_created_valid,2,0.095238,0.181818
0,rule_user_valid,1,0.047619,0.090909
5,rule_tx_id_valid,0,0.000000,0.000000
